<a href="https://colab.research.google.com/github/himanshubhimte69/AgriLite-A-Lightweight-Multi-Crop-Disease-Detector-Across-Diverse-Conditions/blob/main/PlantVillage_VGG16_FineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================
# 1. Kaggle Setup and Dataset Download
# ===============================
from google.colab import files
files.upload()  # Upload kaggle.json first

!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download PlantVillage dataset
!kaggle datasets download -d emmarex/plantdisease
!unzip -q plantdisease.zip -d PlantVillage

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/emmarex/plantdisease
License(s): unknown
100% 658M/658M [00:03<00:00, 50.8MB/s]
100% 658M/658M [00:03<00:00, 182MB/s] 


In [ ]:
# ===============================
# 2. Imports
# ===============================
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing import image_dataset_from_directory

# Parameters
img_size = (224, 224)   # VGG16 expects 224x224
batch_size = 32

In [ ]:
# ===============================
# 3. Dataset Creation
# ===============================
dataset = image_dataset_from_directory(
    "/content/PlantVillage/PlantVillage",
    image_size=img_size,
    batch_size=batch_size,
    validation_split=0.2,
    subset="training",
    seed=123
)

val_dataset = image_dataset_from_directory(
    "/content/PlantVillage/PlantVillage",
    image_size=img_size,
    batch_size=batch_size,
    validation_split=0.2,
    subset="validation",
    seed=123
)

# Split validation set -> validation + test
val_batches = tf.data.experimental.cardinality(val_dataset)
test_dataset = val_dataset.take(val_batches // 2)
val_dataset = val_dataset.skip(val_batches // 2)

class_names = dataset.class_names
num_classes = len(class_names)
print("Classes:", class_names)
print("Num Classes:", num_classes)

Found 20638 files belonging to 15 classes.
Using 16511 files for training.
Found 20638 files belonging to 15 classes.
Using 4127 files for validation.
Classes: ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']
Num Classes: 15


In [ ]:
# ===============================
# 4. Data Augmentation
# ===============================
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

train_ds = dataset.map(lambda x, y: (data_augmentation(x), y))

# Prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_dataset = val_dataset.prefetch(AUTOTUNE)
test_dataset = test_dataset.prefetch(AUTOTUNE)

In [ ]:
# ===============================
# 5. Build VGG16 Model
# ===============================
base_model = VGG16(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # Freeze base initially

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax")
])

# One-hot mapping
def one_hot_map(images, labels):
    return images, tf.one_hot(labels, depth=num_classes)

train_ds_onehot = train_ds.map(one_hot_map)
val_ds_onehot = val_dataset.map(one_hot_map)
test_ds_onehot = test_dataset.map(one_hot_map)

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [ ]:
# ===============================
# 6. Initial Training
# ===============================
history = model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=10
)

Epoch 1/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 284s 519ms/step - accuracy: 0.2795 - loss: 4.1785 - val_accuracy: 0.7023 - val_loss: 1.8154
Epoch 2/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 265s 514ms/step - accuracy: 0.5246 - loss: 2.0291 - val_accuracy: 0.7677 - val_loss: 1.4897
Epoch 3/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 255s 494ms/step - accuracy: 0.5955 - loss: 1.8099 - val_accuracy: 0.8105 - val_loss: 1.3474
Epoch 4/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 270s 510ms/step - accuracy: 0.6384 - loss: 1.6926 - val_accuracy: 0.8264 - val_loss: 1.2648
Epoch 5/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 257s 498ms/step - accuracy: 0.6567 - loss: 1.6038 - val_accuracy: 0.8100 - val_loss: 1.2429
Epoch 6/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 255s 493ms/step - accuracy: 0.6736 - loss: 1.5611 - val_accuracy: 0.8288 - val_loss: 1.1933
Epoch 7/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 257s 498ms/step - accuracy: 0.6709 - loss: 1.5580 - val_accuracy: 0.8134 - val_loss: 1.2071
Epoch 8/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 256s 496ms/step - accuracy: 0.6844 -

In [ ]:
# ===============================
# 7. Fine-Tuning
# ===============================
base_model.trainable = True
fine_tune_at = 10   # Freeze first 10 layers (VGG16 is small, so unfreezing too much may overfit)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

history_finetune = model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=20,
    initial_epoch=10
)

Epoch 11/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 326s 599ms/step - accuracy: 0.7479 - loss: 1.3658 - val_accuracy: 0.8966 - val_loss: 1.0238
Epoch 12/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 348s 561ms/step - accuracy: 0.8396 - loss: 1.1879 - val_accuracy: 0.9226 - val_loss: 0.9708
Epoch 13/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 294s 569ms/step - accuracy: 0.8722 - loss: 1.1215 - val_accuracy: 0.9303 - val_loss: 0.9520
Epoch 14/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 290s 562ms/step - accuracy: 0.8961 - loss: 1.0680 - val_accuracy: 0.9380 - val_loss: 0.9292
Epoch 15/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 290s 561ms/step - accuracy: 0.9098 - loss: 1.0325 - val_accuracy: 0.9418 - val_loss: 0.9140
Epoch 16/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 304s 589ms/step - accuracy: 0.9210 - loss: 1.0071 - val_accuracy: 0.9524 - val_loss: 0.9011
Epoch 17/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 296s 573ms/step - accuracy: 0.9272 - loss: 0.9903 - val_accuracy: 0.9432 - val_loss: 0.8949
Epoch 18/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 316s 560ms/step - accuracy: 

In [ ]:
# ===============================
# 8. Evaluation with Temperature Scaling
# ===============================
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, cohen_kappa_score
from sklearn.preprocessing import label_binarize

TEMPERATURE = 2.0

def predict_with_temperature(model, dataset, T=1.0):
    all_probs, all_labels = [], []
    for images, labels in dataset:
        logits = model(images, training=False)
        scaled_logits = logits / T
        probs = tf.nn.softmax(scaled_logits).numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_probs)

y_true, y_probs = predict_with_temperature(model, test_ds_onehot, T=TEMPERATURE)
y_pred = np.argmax(y_probs, axis=1)

# Convert one-hot labels to integers
y_true_int = np.argmax(y_true, axis=1)

y_true_bin = label_binarize(y_true_int, classes=range(num_classes))

accuracy = accuracy_score(y_true_int, y_pred)
precision = precision_score(y_true_int, y_pred, average="weighted")
recall = recall_score(y_true_int, y_pred, average="weighted")
f1 = f1_score(y_true_int, y_pred, average="weighted")
kappa = cohen_kappa_score(y_true_int, y_pred)
auc = roc_auc_score(y_true_bin, y_probs, average="macro", multi_class="ovr")

print("\n Evaluation Metrics After Calibration:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Kappa    : {kappa:.4f}")


 Evaluation Metrics After Calibration:
Accuracy : 0.9565
Precision: 0.9612
Recall   : 0.9565
F1 Score : 0.9569
AUC      : 0.9991
Kappa    : 0.9524
